In [21]:
import pandas as pd
csv_path = "2/ICD10codes.csv"

In [22]:
pd.read_csv(csv_path)

,A00,0,A000,"Cholera due to Vibrio cholerae 01, biovar cholerae","Cholera due to Vibrio cholerae 01, biovar cholerae.1",Cholera
0,A00,1,A001,"Cholera due to Vibrio cholerae 01, biovar eltor","Cholera due to Vibrio cholerae 01, biovar eltor",Cholera
1,A00,9,A009,"Cholera, unspecified","Cholera, unspecified",Cholera
2,A010,0,A0100,"Typhoid fever, unspecified","Typhoid fever, unspecified",Typhoid fever
3,A010,1,A0101,Typhoid meningitis,Typhoid meningitis,Typhoid fever
4,A010,2,A0102,Typhoid fever with heart involvement,Typhoid fever with heart involvement,Typhoid fever
...,...,...,...,...,...,...
71698,Z991,2,Z9912,Encounter for respirator dependence during pow...,Encounter for respirator [ventilator] dependen...,Dependence on respirator
71699,Z992,NaN,Z992,Dependence on renal dialysis,Dependence on renal dialysis,Dependence on renal dialysis
71700,Z993,NaN,Z993,Dependence on wheelchair,Dependence on wheelchair,Dependence on wheelchair
71701,Z998,1,Z9981,Dependence on supplemental oxygen,Dependence on supplemental oxygen,Dependence on other enabling machines and devices


In [23]:
def normalize_code(code):
    """
    Convert compact ICD-10 to standard format with dot.
    Rules: First 3 chars = category (letter + 2 digits), rest = subcategory
    A000 → A00.0
    A001 → A00.1
    A009 → A00.9
    A0100 → A01.00
    A0101 → A01.01
    A011 → A01.1
    A012 → A01.2
    A020 → A02.0
    T8040xA → T80.40xA (already handled by TXT parser - already has dot)
    """
    code = str(code).strip().upper()
    if not code or code == 'NAN':
        return ''
    
    # Already has dot? Return as-is (TXT file codes like T80.40xA, Y99.0)
    if '.' in code:
        return code
    
    # Compact format: first 3 chars = category, rest = subcategory
    if len(code) >= 3:
        category = code[:3]  # e.g., A00, A01, A02, T80
        subcategory = code[3:]  # e.g., 0, 1, 9, 00, 01, 1, 2, 40xA
        
        if subcategory:
            return f"{category}.{subcategory}"
        else:
            return category  # Just category level (e.g., A00)
    
    return code

In [26]:
def parse_icd10_csv(csv_path):
    """Parse your exact CSV format: 5 columns, comma-separated"""
    # Read with comma separator, no header
    df = pd.read_csv(csv_path, header=None, names=[
        'category', 'subcategory', 'full_code', 'short_desc', 'long_desc','shortest_desc'
    ], dtype=str,
    index_col=False )
    # return df
    # Clean whitespace
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip()
    
    # Handle empty subcategory
    df['subcategory'] = df['subcategory'].replace(['', 'nan', 'None', 'NA'], '')
    
    # Normalize the full_code (column 2) to standard format
    df['code'] = df['full_code'].apply(normalize_code)
    
    # Use long_desc as primary description
    df['description'] = df['long_desc']
    
    print(f"CSV parsed: {len(df)} rows")
    print(f"Sample codes (raw → normalized):")
    for _, row in df.head(15).iterrows():
        print(f"  {row['full_code']} → {row['code']} | {row['description'][:60]}")
    
    print(f"Unique normalized codes: {df['code'].nunique()}")
    return df[['code', 'description', 'category', 'subcategory']].copy()

In [27]:
parse_icd10_csv(csv_path)

CSV parsed: 71704 rows
Sample codes (raw → normalized):
  A000 → A00.0 | Cholera due to Vibrio cholerae 01, biovar cholerae
  A001 → A00.1 | Cholera due to Vibrio cholerae 01, biovar eltor
  A009 → A00.9 | Cholera, unspecified
  A0100 → A01.00 | Typhoid fever, unspecified
  A0101 → A01.01 | Typhoid meningitis
  A0102 → A01.02 | Typhoid fever with heart involvement
  A0103 → A01.03 | Typhoid pneumonia
  A0104 → A01.04 | Typhoid arthritis
  A0105 → A01.05 | Typhoid osteomyelitis
  A0109 → A01.09 | Typhoid fever with other complications
  A011 → A01.1 | Paratyphoid fever A
  A012 → A01.2 | Paratyphoid fever B
  A013 → A01.3 | Paratyphoid fever C
  A014 → A01.4 | Paratyphoid fever, unspecified
  A020 → A02.0 | Salmonella enteritis
Unique normalized codes: 71704


,code,description,category,subcategory
0,A00.0,"Cholera due to Vibrio cholerae 01, biovar chol...",A00,0
1,A00.1,"Cholera due to Vibrio cholerae 01, biovar eltor",A00,1
2,A00.9,"Cholera, unspecified",A00,9
3,A01.00,"Typhoid fever, unspecified",A010,0
4,A01.01,Typhoid meningitis,A010,1
...,...,...,...,...
71699,Z99.12,Encounter for respirator [ventilator] dependen...,Z991,2
71700,Z99.2,Dependence on renal dialysis,Z992,NaN
71701,Z99.3,Dependence on wheelchair,Z993,NaN
71702,Z99.81,Dependence on supplemental oxygen,Z998,1
